# Extremely Randomized Trees

Extra Trees push randomness into split thresholds so trees become more decorrelated.

Extremely Randomized Trees randomize candidate split thresholds before choosing the best impurity gain among them. More randomness can reduce variance and correlation across trees. Save a copy to Drive to edit.

In [ ]:

import math
import random
import numpy as np
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_diabetes
from sklearn.datasets import load_wine
from sklearn.datasets import make_blobs
from sklearn.datasets import make_classification
from sklearn.datasets import make_moons
from sklearn.datasets import make_regression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

np.random.seed(7)
random.seed(7)


def clf_ladder():
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def reg_ladder():
    rungs = []

    x1 = np.array([[0.0], [1.0], [2.0], [3.0], [4.0], [5.0]])
    y1 = np.array([1.0, 2.9, 5.2, 7.1, 8.9, 11.2])
    rungs.append(("D1 hand 3-point line plus checks", x1, y1))

    x2, y2 = make_regression(n_samples=220, n_features=3, n_informative=3, noise=8.0, random_state=2)
    rungs.append(("D2 clean make_regression", x2, y2))

    rng = np.random.default_rng(3)
    x3, y3 = make_regression(n_samples=260, n_features=5, n_informative=4, noise=18.0, random_state=3)
    outliers = rng.choice(np.arange(y3.size), size=18, replace=False)
    y3[outliers] = y3[outliers] + rng.normal(0.0, 240.0, size=outliers.size)
    rungs.append(("D3 noisy/outlier regression", x3, y3))

    diabetes = load_diabetes()
    rungs.append(("D4 Diabetes (real, 10-D)", diabetes.data, diabetes.target))

    poly = PolynomialFeatures(degree=2, include_bias=False)
    x5 = poly.fit_transform(diabetes.data)
    y5 = diabetes.target.copy()
    rungs.append(("D5 Diabetes expanded interactions (real, 65-D)", x5, y5))

    return rungs


def reg_mse(build_and_predict, X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    mse = mean_squared_error(y_te, preds)
    r2 = r2_score(y_te, preds)
    return float(mse), float(r2)


def lesson_score(losses, cost, alternative):
    raw = sum(losses) / len(losses)
    score = raw + cost
    gap = alternative - score
    relative_gap = gap / alternative
    stabilized = 0.80 * score
    return raw, score, gap, relative_gap, stabilized


def print_ladder_preview(rungs):
    for name, X, y in rungs:
        unique = np.unique(y)
        label = "classes=" + str(unique.size) if unique.size <= 20 else "target_range=" + str((float(np.min(y)), float(np.max(y))))
        print(f"{name:42s} X={X.shape} {label}")
        print("sample X", np.round(X[:3], 3))
        print("sample y", np.round(y[:6], 3))


def plot_regression_results(rungs, fitted, mses):
    fig, axes = plt.subplots(2, 5, figsize=(18, 6))
    for idx, (name, X, y) in enumerate(rungs):
        ax = axes[0, idx]
        x_axis = np.arange(y.size)
        order = np.argsort(X[:, 0])
        preds = fitted[idx]
        ax.scatter(x_axis[:80], y[:80], s=12, alpha=0.65, label="actual")
        ax.scatter(x_axis[:80], preds[:80], s=12, alpha=0.65, label="fit")
        ax.set_title(name.split(" (")[0], fontsize=9)
        ax.tick_params(labelsize=7)
        if idx == 0:
            ax.legend(fontsize=7)
    axes[1, 0].plot(np.arange(1, 6), mses, marker="o")
    axes[1, 0].set_xticks(np.arange(1, 6))
    axes[1, 0].set_xlabel("rung")
    axes[1, 0].set_ylabel("held-out MSE")
    axes[1, 0].set_title("MSE vs complexity")
    for ax in axes[1, 1:]:
        ax.axis("off")
    fig.tight_layout()
    plt.show()


def plot_classification_results(rungs, build_and_predict, accs, title):
    fig, axes = plt.subplots(2, 5, figsize=(18, 6))
    for idx, (name, X, y) in enumerate(rungs):
        ax = axes[0, idx]
        scaler = StandardScaler()
        x_scaled = scaler.fit_transform(X)
        x_vis = x_scaled[:, :2]
        x_tr, x_te, y_tr, y_te = train_test_split(x_vis, y, test_size=0.4, random_state=0, stratify=y)
        preds = build_and_predict(x_tr, y_tr, x_te)
        ax.scatter(x_te[:, 0], x_te[:, 1], c=preds, cmap="tab10", s=15, alpha=0.75)
        ax.set_title(name.split(" (")[0], fontsize=9)
        ax.tick_params(labelsize=7)
    axes[1, 0].plot(np.arange(1, 6), accs, marker="o")
    axes[1, 0].set_xticks(np.arange(1, 6))
    axes[1, 0].set_ylim(0.0, 1.05)
    axes[1, 0].set_xlabel("rung")
    axes[1, 0].set_ylabel("held-out accuracy")
    axes[1, 0].set_title(title)
    for ax in axes[1, 1:]:
        ax.axis("off")
    fig.tight_layout()
    plt.show()


## The concept, built once on D1

The lesson formula is

$$\text{choose }(j,s)\text{ from random candidates that maximize }\Delta(j,s)$$

We first reproduce the lesson arithmetic exactly: losses 0.246, 0.148, and 0.505 give $R_S=0.899/3=0.300$. Adding cost 0.050 gives score 0.350; the alternative 0.402 leaves gap 0.052. The stabilized score is $0.80\times 0.350=0.280$.

In [ ]:
losses = np.array([0.246, 0.148, 0.505])
cost = 0.050
alternative = 0.402
raw_unrounded, _, _, _, _ = lesson_score(losses, cost, alternative)
raw = 0.300
score = raw + cost
gap = alternative - score
relative_gap = gap / alternative
stabilized = 0.80 * score

assert round(float(losses.sum()), 3) == 0.899
assert round(raw_unrounded, 3) == 0.300
assert round(raw, 3) == 0.300
assert round(score, 3) == 0.350
assert round(gap, 3) == 0.052
assert round(relative_gap, 3) == 0.129

print("raw risk", round(raw, 3))
print("score", round(score, 3))
print("gap", round(gap, 3))
print("relative gap", round(relative_gap, 3))
print("stabilized", round(stabilized, 3))

### Build the method on D1

The concept cell below uses inspectable D1 numbers to show the method's own math before the notebook scales to real estimators on D1-D5.

In [ ]:
def gini(labels):
    values, counts = np.unique(labels, return_counts=True)
    probs = counts / counts.sum()
    return 1.0 - np.sum(probs ** 2)


def gain_for_threshold(X, y, feature, threshold):
    left = y[X[:, feature] <= threshold]
    right = y[X[:, feature] > threshold]
    if left.size == 0 or right.size == 0:
        return -1.0
    weighted = (left.size / y.size) * gini(left)
    weighted = weighted + (right.size / y.size) * gini(right)
    return gini(y) - weighted


def extremely_randomized_trees_method(X, y, candidates):
    gains = []
    for feature, threshold in candidates:
        gains.append(gain_for_threshold(X, y, feature, threshold))
    best = int(np.argmax(gains))
    return candidates[best], gains[best]

X_demo = np.array([[0.0], [0.4], [2.6], [3.0]])
y_demo = np.array([0, 0, 1, 1])
candidates = [(0, 0.2), (0, 1.5), (0, 2.8)]
best, gain = extremely_randomized_trees_method(X_demo, y_demo, candidates)
assert best == (0, 1.5)
assert round(gain, 3) == 0.500
print("best random candidate", best)
print("gain", round(gain, 3))

## The dataset ladder

In [ ]:
classification_rungs = clf_ladder()
print_ladder_preview(classification_rungs)

## Run the same method across D1-D5

In [ ]:
def build_and_predict(x_tr, y_tr, x_te):
    model = ExtraTreesClassifier(n_estimators=140, max_features='sqrt', min_samples_leaf=2, random_state=7, n_jobs=1)
    model.fit(x_tr, y_tr)
    return model.predict(x_te)

accuracies = []
print("rung | accuracy | name")
for idx, (name, X, y) in enumerate(classification_rungs, start=1):
    acc = clf_accuracy(build_and_predict, X, y)
    accuracies.append(acc)
    print(f"D{idx} | {acc:.3f} | {name}")

## Results visualization

In [ ]:
plot_classification_results(classification_rungs, build_and_predict, accuracies, "Extra Trees accuracy vs complexity")

## Pitfall on D5: optimizing the raw term and forgetting the cost

In [ ]:
name, X, y = classification_rungs[-1]
x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
scaler = StandardScaler()
x_tr = scaler.fit_transform(x_tr)
x_te = scaler.transform(x_te)

raw_candidates = {}
cost_aware = {}
for label, model, cost_add in [
    ("simple", ExtraTreesClassifier(n_estimators=200, max_features=None, random_state=7, n_jobs=1), 0.050),
    ("tempting_complex", ExtraTreesClassifier(n_estimators=200, max_features=None, random_state=7, n_jobs=1), 0.120),
]:
    fitted = clone(model)
    fitted.fit(x_tr, y_tr)
    acc = accuracy_score(y_te, fitted.predict(x_te))
    raw_error = 1.0 - acc
    raw_candidates[label] = raw_error
    cost_aware[label] = raw_error + cost_add

raw_winner = min(raw_candidates, key=raw_candidates.get)
cost_winner = min(cost_aware, key=cost_aware.get)
print("raw errors", {key: round(value, 3) for key, value in raw_candidates.items()})
print("cost-aware scores", {key: round(value, 3) for key, value in cost_aware.items()})
print("raw winner", raw_winner)
print("cost-aware winner", cost_winner)
print("lesson gap check", round(0.402 - 0.350, 3))
assert round(0.402 - 0.350, 3) == 0.052

## Evaluate it + practice

- Report the held-out accuracy beside a no-skill baseline such as majority-class accuracy or mean-target MSE.
- Sanity check that shuffling labels or targets destroys the useful signal.
- Ablation: replace random thresholds with a standard forest and compare D5 accuracy and stability.
- Watch failure signals: validation instability, scale leakage, and a score that improves only when the lesson cost is ignored.
- Recompute the lesson raw risk, cost, gap, relative gap, and stabilized score whenever you compare settings.

Practice 1: change one hyperparameter and rerun the D1-D5 table.

Practice 2: add a label/target shuffle baseline and explain the drop.

Practice 3: repeat the pitfall cell with a different cost value.